In [2]:
import json, random, time
from datetime import datetime, timezone
from confluent_kafka import Producer

BOOTSTRAP = "localhost:9092,localhost:9094,localhost:9096"
TOPIC = "urbanpulse.traffic_signals"
ZONES = [f"ZONE-{z}" for z in "ABCDEFGH"]
JUNCTIONS_PER_ZONE = 475
GRIDLOCK_PROBABILITY = 0.03

producer = Producer({
    "bootstrap.servers": BOOTSTRAP, "acks": "all",
    "enable.idempotence": True, "retries": 5, "linger.ms": 5,
})

JUNCTION_IDS = {zone: [f"{zone}-JN-{i:04d}" for i in range(JUNCTIONS_PER_ZONE)] for zone in ZONES}

def make_event(zone, junction_id):
    gridlock = random.random() < GRIDLOCK_PROBABILITY
    avg_wait = round(random.uniform(180, 260), 1) if gridlock else round(random.uniform(15, 90), 1)
    return {
        "junction_id": junction_id, "zone": zone, "avg_wait_sec": avg_wait,
        "vehicle_count": random.randint(5, 120),
        "signal_phase": random.choice(["RED", "GREEN", "YELLOW"]),
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }

def run(duration_sec=30, target_rate=100):
    end_time = time.time() + duration_sec
    sent = 0
    while time.time() < end_time:
        zone = random.choice(ZONES)
        junction_id = random.choice(JUNCTION_IDS[zone])
        event = make_event(zone, junction_id)
        producer.produce(TOPIC, key=junction_id.encode(), value=json.dumps(event).encode())
        producer.poll(0)
        sent += 1
        time.sleep(1.0 / target_rate)
    producer.flush(10)
    print(f"[traffic_signals] done. total sent={sent}")

run(duration_sec=300)  # run longer here (5 min) so consumers below have time to show lag divergence

[traffic_signals] done. total sent=28848
